# Best Model — Churn Classification

Trains only the winning model found by the AI Data Science Agent's Ralph Loop for this dataset/objective:

| | |
|---|---|
| **Dataset** | `workspace/datasets/sample_churn.csv` |
| **Objective** | Predict whether a customer will churn |
| **Task** | Classification |
| **Target column** | `churn` |
| **Metric** | F1 (weighted) |
| **Winning model** | Random Forest (`sklearn.ensemble.RandomForestClassifier`) |
| **Hyperparameters** | defaults (`n_estimators=100`, `max_depth=None`, `random_state=42`) |
| **Feature engineering** | one-hot encode categoricals, median-fill numeric NaNs |
| **Best F1 achieved** | 0.8526 |

This notebook is self-contained — it does not import anything from the agent's `app/`/`tools/` package, so it will run on its own outside this project.

## 1. Imports

In [ ]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

## 2. Load the dataset

In [ ]:
DATASET_PATH = "workspace/datasets/sample_churn.csv"  # adjust if you moved this notebook
TARGET = "churn"

df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=[TARGET])

print(df.shape)
df.head()

## 3. Prepare features and target

Same feature engineering the winning run used: one-hot encode categorical columns, then median-fill any missing numeric values.

In [ ]:
y = df[TARGET]
X = df.drop(columns=[TARGET])

X = pd.get_dummies(X, drop_first=True)

numeric_cols = X.select_dtypes(include="number").columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

X.head()

## 4. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Train the winning model

Random Forest with the default hyperparameters the winning experiment used.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
)
model.fit(X_train, y_train)

## 6. Evaluate

In [ ]:
preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, average="weighted", zero_division=0)
recall = recall_score(y_test, preds, average="weighted", zero_division=0)
f1 = f1_score(y_test, preds, average="weighted", zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print()
print(classification_report(y_test, preds))

In [ ]:
confusion_matrix(y_test, preds)

## 7. Feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances

## 8. Save the trained model

In [ ]:
MODEL_PATH = "best_model.joblib"
Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f"Saved to {MODEL_PATH}")

## 9. Load it back and predict (sanity check)

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
loaded_model.predict(X_test.head())